**Problem Definition**

**Dataset Selected:**
The PlantVillage Dataset, which is widely used for agricultural plant disease detection.

**Topic Representation:**
This dataset represents the critical agricultural issue of crop health monitoring. It contains thousands of images of various plant leaves, categorized by crop species and their respective health status (either healthy or infected with a specific disease).

**Analytical Questions:**

Crop Vulnerability: Which crop species in the dataset exhibit the widest variety of diseases, and what does this suggest about their susceptibility?

Health Distribution: What is the overall ratio of healthy versus diseased plants in this dataset, and how does this distribution vary across different crops?

Data Quality & Consistency: Are there any statistical anomalies or outliers in the dataset structure (e.g., filename patterns) that could indicate inconsistencies in data collection?

**Why These Questions Matter:**
Understanding which crops are more prone to a variety of diseases helps in prioritizing agricultural research and resource allocation. Additionally, analyzing the distribution and quality of the data ensures that any insights or future machine learning models built on this dataset are reliable and based on a balanced understanding of the real-world agricultural context.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os 

In [3]:
base_path = r'E:\(1) PDF\10th Semeaster\PROGRAMMING IN PYTHON\python-finalterm-project\Dataset\plantvillage dataset\color'

if os.path.exists(base_path):
    all_data = []
    for folder in os.listdir(base_path):
        folder_path = os.path.join(base_path, folder)
        if os.path.isdir(folder_path):
            for file in os.listdir(folder_path):
                if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                    all_data.append({
                        'Image_Name': file,
                        'Disease_Type': folder
                    })
    df = pd.DataFrame(all_data)
    print("Successfully loaded data!")
    print(f"Total Images: {len(df)}")
else:
    print("Path not found! Please check the folder path.")

Successfully loaded data!
Total Images: 54305


## 4.2 Data Understanding
Markdown Cell:

In this section, we explore the initial structure and quality of the dataset. This includes checking the dataset's dimensions, data types, and identifying any missing values to ensure the data is ready for analysis.

In [5]:
# 1. Dataset Shape
print("--- Dataset Shape ---")
print(f"The dataset contains {df.shape[0]} rows and {df.shape[1]} columns.")

--- Dataset Shape ---
The dataset contains 54305 rows and 2 columns.


In [6]:
# 2. Data Types and Structure
print("\n--- Data Structure & Types ---")
df.info()


--- Data Structure & Types ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54305 entries, 0 to 54304
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Image_Name    54305 non-null  object
 1   Disease_Type  54305 non-null  object
dtypes: object(2)
memory usage: 848.6+ KB


In [7]:
print("\n--- Sample Data ---")
display(df.head())


--- Sample Data ---


,Image_Name,Disease_Type
0,00075aa8-d81a-4184-8541-b692b78d398a___FREC_Sc...,Apple___Apple_scab
1,01a66316-0e98-4d3b-a56f-d78752cd043f___FREC_Sc...,Apple___Apple_scab
2,01f3deaa-6143-4b6c-9c22-620a46d8be04___FREC_Sc...,Apple___Apple_scab
3,0208f4eb-45a4-4399-904e-989ac2c6257c___FREC_Sc...,Apple___Apple_scab
4,023123cb-7b69-4c9f-a521-766d7c8543bb___FREC_Sc...,Apple___Apple_scab


In [8]:
# 4. Summary Statistics (Categorical)
print("\n--- Summary Statistics ---")
print(df.describe(include='all'))


--- Summary Statistics ---
                                               Image_Name  \
count                                               54305   
unique                                              54305   
top     fff6bf83-797c-4de4-83f7-39b74786fec4___UF.GRC_...   
freq                                                    1   

                                    Disease_Type  
count                                      54305  
unique                                        38  
top     Orange___Haunglongbing_(Citrus_greening)  
freq                                        5507  


In [9]:
# 5. Missing-Value Summary
print("\n--- Missing Values Summary ---")
missing_values = df.isnull().sum()
print(missing_values)


--- Missing Values Summary ---
Image_Name      0
Disease_Type    0
dtype: int64


## Interpretation of Data Understanding Outputs
Based on the initial inspection of the dataset, here is a summary of its structure and quality:

**Structure:** The dataset primarily consists of two columns: image_name (representing the unique file identifiers) and disease_type (which contains combined information about the plant species and its specific health condition).

**Dimensions:** It contains over 54,300 rows, which easily surpasses the project requirement of at least 1,000 rows, providing a substantial amount of data for robust analysis.

**Data Quality:** The isnull().sum() function returned a result of 0 for all columns, confirming that there are no empty cells or missing values within the dataset, ensuring high data integrity.

**Categories:** The disease_type column contains categorical data with 38 unique classes, ranging from healthy leaves to those infected with various diseases (e.g., Apple Healthy, Tomato Early Blight), allowing for detailed subgroup comparisons.

## 4.3 Data Cleaning and Preparation
In this stage, the raw dataset was processed to ensure data integrity and reliability for subsequent analysis. The following cleaning steps were performed:

1. Handling Duplicates
The Problem: The dataset may contain redundant entries where the same image is listed multiple times.

Action Taken: Identified and removed all duplicate rows using the drop_duplicates() function.

Justification: Eliminating duplicates is essential to maintain an accurate count of images. This ensures that our statistical distribution of plant diseases is not biased by repeated data points.

In [10]:
## 4.3 Data Cleaning and Preparation

initial_rows = len(df)
df = df.drop_duplicates()
final_rows = len(df)

print(f"--- Step 1: Handling Duplicates ---")
print(f"Initial rows: {initial_rows}")
print(f"Rows after removing duplicates: {final_rows}")
print(f"Justification: Removed {initial_rows - final_rows} duplicate entries")

--- Step 1: Handling Duplicates ---
Initial rows: 54305
Rows after removing duplicates: 54305
Justification: Removed 0 duplicate entries


## Column Name Standardization
The Problem: Original column names might contain spaces or inconsistent capitalization, which can lead to syntax errors during coding.

Action Taken: Standardized all column names to lowercase snake_case (e.g., image_name, disease_type).

Justification: This follows Python's PEP 8 naming conventions. Standardized names make the code more readable, consistent, and easier to debug during attribute access and filtering.

In [11]:

df.columns = [col.replace(' ', '_').lower() for col in df.columns]
print(f"\n--- Step 2: Renaming Columns ---")
print(f"New column names: {df.columns.tolist()}")
print(f"Justification: Standardizing column names")


--- Step 2: Renaming Columns ---
New column names: ['image_name', 'disease_type']
Justification: Standardizing column names


## Text Data Validation and Stripping
The Problem: Categorical strings often contain leading or trailing whitespaces due to folder naming inconsistencies.

Action Taken: Applied the str.strip() method to all categorical text entries.

Justification: This prevents logical errors during grouping and aggregation. For instance, it ensures that "Apple" and "Apple " are treated as the same category, maintaining the accuracy of our species-wise analysis.

In [12]:

df['disease_type'] = df['disease_type'].str.strip()
print(f"\n--- Step 3: Data Validation & Stripping ---")
print(f"Justification: Performed string stripping")


--- Step 3: Data Validation & Stripping ---
Justification: Performed string stripping


In [13]:
print("\nCleaning complete! Current Dataset Head:")
display(df.head())


Cleaning complete! Current Dataset Head:


,image_name,disease_type
0,00075aa8-d81a-4184-8541-b692b78d398a___FREC_Sc...,Apple___Apple_scab
1,01a66316-0e98-4d3b-a56f-d78752cd043f___FREC_Sc...,Apple___Apple_scab
2,01f3deaa-6143-4b6c-9c22-620a46d8be04___FREC_Sc...,Apple___Apple_scab
3,0208f4eb-45a4-4399-904e-989ac2c6257c___FREC_Sc...,Apple___Apple_scab
4,023123cb-7b69-4c9f-a521-766d7c8543bb___FREC_Sc...,Apple___Apple_scab


In [14]:
display(df.tail())

,image_name,disease_type
54300,ffb295c9-f14e-4a15-831a-bf905da7fcb6___UF.GRC_...,Tomato___Tomato_Yellow_Leaf_Curl_Virus
54301,ffe08ccc-c55e-4ca2-9234-2906b98b8d05___YLCV_NR...,Tomato___Tomato_Yellow_Leaf_Curl_Virus
54302,ffe996e5-c8dc-47b7-bca2-4fc25e5ac57c___UF.GRC_...,Tomato___Tomato_Yellow_Leaf_Curl_Virus
54303,fff42f1b-7ec4-46e3-9269-45932e63635e___YLCV_GC...,Tomato___Tomato_Yellow_Leaf_Curl_Virus
54304,fff6bf83-797c-4de4-83f7-39b74786fec4___UF.GRC_...,Tomato___Tomato_Yellow_Leaf_Curl_Virus


## Justification for Feature Engineering:

**Crop & Health Status:** The original disease_type column was complex. We derived crop and health_status to perform subgroup comparisons, which is a core requirement of this project.

**Name Length:** We created a numerical feature name_length using the image_name column. This allows us to perform NumPy-based statistical computations and outlier detection, ensuring we meet the analytical depth required.

In [15]:

df['crop'] = df['disease_type'].apply(lambda x: str(x).split('___')[0] if '___' in str(x) else str(x).split(' | ')[0])


df['health_status'] = df['disease_type'].apply(lambda x: 'Healthy' if 'healthy' in str(x).lower() else 'Diseased')


df['name_length'] = df['image_name'].apply(len)


df['disease_type'] = df['disease_type'].str.replace('___', ' | ').str.replace('_', ' ')

print("--- Feature Engineering Complete ---")
print(f"New Columns Created: ['crop', 'health_status', 'name_length']")
print(f"Justification: Created 'crop' and 'health_status' for subgroup comparison, and 'name_length' to perform statistical numerical analysis.")

display(df.head())

--- Feature Engineering Complete ---
New Columns Created: ['crop', 'health_status', 'name_length']
Justification: Created 'crop' and 'health_status' for subgroup comparison, and 'name_length' to perform statistical numerical analysis.


,image_name,disease_type,crop,health_status,name_length
0,00075aa8-d81a-4184-8541-b692b78d398a___FREC_Sc...,Apple | Apple scab,Apple,Diseased,57
1,01a66316-0e98-4d3b-a56f-d78752cd043f___FREC_Sc...,Apple | Apple scab,Apple,Diseased,57
2,01f3deaa-6143-4b6c-9c22-620a46d8be04___FREC_Sc...,Apple | Apple scab,Apple,Diseased,57
3,0208f4eb-45a4-4399-904e-989ac2c6257c___FREC_Sc...,Apple | Apple scab,Apple,Diseased,57
4,023123cb-7b69-4c9f-a521-766d7c8543bb___FREC_Sc...,Apple | Apple scab,Apple,Diseased,57


In [16]:

print("--- Samples of Healthy and Diseased Plants ---")

samples = pd.concat([
    df[df['health_status'] == 'Healthy'].head(2),
    df[df['health_status'] == 'Diseased'].head(2)
])

display(samples)




--- Samples of Healthy and Diseased Plants ---


,image_name,disease_type,crop,health_status,name_length
1526,0055dd26-23a7-4415-ac61-e0b44ebfaf80___RS_HL 5...,Apple | healthy,Apple,Healthy,53
1527,00907d8b-6ae6-4306-bfd7-d54471981a86___RS_HL 5...,Apple | healthy,Apple,Healthy,53
0,00075aa8-d81a-4184-8541-b692b78d398a___FREC_Sc...,Apple | Apple scab,Apple,Diseased,57
1,01a66316-0e98-4d3b-a56f-d78752cd043f___FREC_Sc...,Apple | Apple scab,Apple,Diseased,57


In [17]:
print(f"\nTotal Healthy Images found: {len(df[df['health_status'] == 'Healthy'])}")
print(f"Total Diseased Images found: {len(df[df['health_status'] == 'Diseased'])}")


Total Healthy Images found: 15084
Total Diseased Images found: 39221


## Data Analysis Strategy:

*Subgroup Comparison:* Used groupby to compare healthy vs. diseased counts across different plant species.

*NumPy-based Statistics:* Leveraged NumPy to calculate the mean and standard deviation of filename lengths to establish a baseline for data consistency.

*Outlier Detection:* Used Z-scores to identify statistical anomalies in the dataset structure, which helps in identifying potentially corrupted or mislabeled entries.

*Relationship Analysis:* Examined the correlation between the volume of images collected and the variety of diseases identified for each crop.

In [20]:
#  Subgroup Comparison
crop_health_analysis = df.groupby(['crop', 'health_status']).size().unstack().fillna(0)
print("---  Subgroup Comparison (Crop vs Health) ---")
print(crop_health_analysis)

---  Subgroup Comparison (Crop vs Health) ---
health_status            Diseased  Healthy
crop                                      
Apple                      1526.0   1645.0
Blueberry                     0.0   1502.0
Cherry_(including_sour)    1052.0    854.0
Corn_(maize)               2690.0   1162.0
Grape                      3639.0    423.0
Orange                     5507.0      0.0
Peach                      2297.0    360.0
Pepper,_bell                997.0   1478.0
Potato                     2000.0    152.0
Raspberry                     0.0    371.0
Soybean                       0.0   5090.0
Squash                     1835.0      0.0
Strawberry                 1109.0    456.0
Tomato                    16569.0   1591.0


In [21]:
# Ranking
disease_variety = df.groupby('crop')['disease_type'].nunique().sort_values(ascending=False)
print("\n---  Ranking: Crops with Most Disease Varieties ---")
print(disease_variety)


---  Ranking: Crops with Most Disease Varieties ---
crop
Tomato                     10
Apple                       4
Grape                       4
Corn_(maize)                4
Potato                      3
Cherry_(including_sour)     2
Strawberry                  2
Peach                       2
Pepper,_bell                2
Blueberry                   1
Orange                      1
Raspberry                   1
Squash                      1
Soybean                     1
Name: disease_type, dtype: int64
